# 容量约束设施选址问题(CFLP)

**类别：** 选址

来源：[https://www.hexaly.com/templates/capacitated-facility-location-problem-cflp](https://www.hexaly.com/templates/capacitated-facility-location-problem-cflp)


## 问题描述

**在容量约束设施选址问题(Capacitated Facility Location Problem, CFLP)**中,若干具有已知需求的客户点必须被分配给可用的设施。分配给每个设施的客户点总需求不得超过其容量。每个设施都有一个固定的开启费用,只要该设施至少服务一个客户点就必须支付。目标是最小化总成本,该成本为所有已开启设施的开启费用与所有客户点的分配费用之和。

	

### 学习要点

- 添加 [set decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 以建模分配给每个设施的客户点
- 使用 [lambda 表达式](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 计算每个设施的总需求与总费用


## 数据

所提供的数据文件来自 [OR-LIB](http://people.brunel.ac.uk/~mastjjb/jeb/orlib/pmedcapinfo.html)。数据格式如下:

- 第一行:候选设施数量与客户点数量
- 每个设施的容量与开启费用
- 每个客户点的需求
- 每个设施与每个客户点之间的分配费用


## 建模方法

容量约束设施选址问题(CFLP)的 Hexaly 模型使用 [set decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)来表示分配给每个设施的客户点。借助 [**partition** 算子](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html),我们确保每个客户点恰好被分配到一个设施。

我们可以使用需求数组上的 **at** 算子来访问序列中每个客户点的需求。每个设施所服务的总需求通过一个 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 计算,该函数将 **sum** 算子应用于所有关联客户点的需求。该总需求必须不超过该设施的容量。

类似地,我们将每个设施的分配费用计算为其所服务的所有客户点的分配价格之和。使用 **count** 算子,我们检查每个设施是否至少服务一个客户点。如果是,则还需支付该设施的开启费用。

目标函数为所有设施的开启费用与分配费用之和。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys


def main(instanceFile, strTimeLimit, solFile):

    #
    # Read instance data
    #
    nb_max_facilities, nb_sites, capacity_data, opening_price_data, \
        demand_data, allocation_price_data = read_data(instanceFile)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Facilities are represented by the set of sites they provide
        facility_assignments = [model.set(nb_sites) for _ in range(nb_max_facilities)]

        # Each site is covered by exactly one facility
        model.constraint(model.partition(facility_assignments))

        # Converting demand and allocationPrice into Hexaly array
        demand = model.array(demand_data)
        allocation_price = model.array(allocation_price_data)

        cost = [None] * nb_max_facilities
        for f in range(nb_max_facilities):
            facility = facility_assignments[f]
            size = model.count(facility)

            # Capacity constraint
            demand_lambda = model.lambda_function(lambda i: demand[i])
            model.constraint(model.sum(facility, demand_lambda) <= capacity_data[f])

            # Cost (allocation price + opening price)
            costSelector = model.lambda_function(lambda i: model.at(allocation_price, f, i))
            cost[f] = model.sum(facility, costSelector) + opening_price_data[f] * (size > 0)

        # Objective : minimize total cost
        totalCost = model.sum(cost)
        model.minimize(totalCost)

        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = int(strTimeLimit)

        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        # - value of the objective
        # - indices of the open facilities followed by all the sites they provide
        #
        if solFile:
            with open(solFile, 'w') as outputFile:
                outputFile.write("%d" % totalCost.value)
                for f in range(nb_max_facilities):
                    if cost[f].value > 0:
                        outputFile.write("%d\n" % f)
                        for site in facility_assignments[f].value:
                            outputFile.write("%d " % site)
                        outputFile.write("\n")


def read_elem(filename):
    with open(filename) as f:
        return [str(elem) for elem in f.read().split()]


def read_data(filename):
    file_it = iter(read_elem(filename))

    nb_max_facilities = int(next(file_it))
    nb_sites = int(next(file_it))

    capacity_data = []
    opening_price_data = []
    demand_data = []
    allocation_price_data = []

    for f in range(nb_max_facilities):
        # List of facilities capacities
        capacity_data.append(float(next(file_it)))
        # List of fixed costs induced by the facilities opening
        opening_price_data.append(float(next(file_it)))
        allocation_price_data.append([])

    # Demand of each site
    for s in range(nb_sites):
        demand_data.append(float(next(file_it)))

    # Allocation price between sites and facilities
    for f in range(nb_max_facilities):
        for s in range(nb_sites):
            allocation_price_data[f].append(float(next(file_it)))

    return nb_max_facilities, nb_sites, capacity_data, opening_price_data, \
        demand_data, allocation_price_data


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python capacitated_facility_location.py input_file \
            [output_file] [time_limit]")
        sys.exit(1)

    instanceFile = sys.argv[1]
    solFile = sys.argv[2] if len(sys.argv) > 2 else None
    strTimeLimit = sys.argv[3] if len(sys.argv) > 3 else "20"

    main(instanceFile, strTimeLimit, solFile)
